In [3]:
import torch
import torchvision

resnet = torchvision.models.resnet18(pretrained=True)

In [4]:
example_input = torch.rand(1, 3, 224, 224) 
trace_model = torch.jit.trace(resnet, example_input)

In [6]:
import urllib
label_url = 'https://storage.googleapis.com/download.tensorflow.org/data/ImageNetLabels.txt'
class_labels = urllib.request.urlopen(label_url).read().decode("utf-8").splitlines()
class_labels = class_labels[1:] # remove the first class which is background
assert len(class_labels) == 1000

In [8]:
# Set the image scale and bias for input image preprocessing.
import coremltools as ct
scale = 1/(0.226*255.0)
bias = [- 0.485/(0.229) , - 0.456/(0.224), - 0.406/(0.225)]

image_input = ct.ImageType(name="input_1",
                           shape=example_input.shape,
                           scale=scale, bias=bias)

In [10]:
# Using image_input in the inputs parameter:
# Convert to Core ML using the Unified Conversion API.
model = ct.convert(
    trace_model,
    inputs=[image_input],
    classifier_config = ct.ClassifierConfig(class_labels),
    compute_units=ct.ComputeUnit.CPU_ONLY,
)

Running MIL Common passes:   0%|                    | 0/34 [00:00<?, ? passes/s]/opt/homebrew/anaconda3/envs/py39/lib/python3.9/site-packages/coremltools/converters/mil/mil/passes/name_sanitization_utils.py:129: UserWarning: Output, '363', of the source model, has been renamed to 'var_363' in the Core ML model.
  warnings.warn(msg.format(var.name, new_name))
Running MIL Clean up passes: 100%|██████████| 9/9 [00:00<00:00, 160.16 passes/s]
Translating MIL ==> NeuralNetwork Ops: 100%|█| 509/509 [00:00<00:00, 641.76 ops/


In [11]:
model.save("resnet18.mlmodel")

In [14]:
coreml_model_path = "./resnet18.mlmodel"
spec = ct.utils.load_spec(coreml_model_path)
builder = ct.models.neural_network.NeuralNetworkBuilder(spec=spec)
builder.inspect_layers()

[Id: 228], Name: 363 (Type: innerProduct)
          Updatable: False
          Input blobs: ['input']
          Output blobs: ['var_363']
[Id: 227], Name: input (Type: reshapeStatic)
          Updatable: False
          Input blobs: ['x']
          Output blobs: ['input']
[Id: 226], Name: x (Type: pooling)
          Updatable: False
          Input blobs: ['input.56']
          Output blobs: ['x']
[Id: 225], Name: input.56 (Type: activation)
          Updatable: False
          Input blobs: ['input.55']
          Output blobs: ['input.56']
[Id: 224], Name: input.55 (Type: add)
          Updatable: False
          Input blobs: ['out', 'input.50']
          Output blobs: ['input.55']
[Id: 223], Name: out (Type: batchnorm)
          Updatable: False
          Input blobs: ['out_div']
          Output blobs: ['out']
[Id: 222], Name: out_div (Type: divideBroadcastable)
          Updatable: False
          Input blobs: ['sub_19', 'sqrt_19']
          Output blobs: ['out_div']
[Id: 221], Name

In [19]:
builder.inspect_input_features()

[Id: 0] Name: input_1
          Type: imageType {
  width: 224
  height: 224
  colorSpace: RGB
}

